In [1]:
!pip -q install -U gradio ultralytics opencv-python

import torch
import torch.nn.functional as F
import numpy as np
import cv2
from PIL import Image
import gradio as gr
from ultralytics import YOLO
import torch.nn as nn
from pathlib import Path
import math
import os
import pandas as pd
import numpy as np
from skimage.metrics import structural_similarity as ssim_fn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 107.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.7/55.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 73.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [12]:
!unzip /content/drive/MyDrive/Restormer-Detection-Classfication/Detection/runs_curetsd_single.zip

Archive:  /content/drive/MyDrive/Restormer-Detection-Classfication/Detection/runs_curetsd_single.zip
   creating: runs_curetsd_single/runs_curetsd_single/
   creating: runs_curetsd_single/runs_curetsd_single/yolov8n_singleclass_img1024/
  inflating: runs_curetsd_single/runs_curetsd_single/yolov8n_singleclass_img1024/train_batch22052.jpg  
  inflating: runs_curetsd_single/runs_curetsd_single/yolov8n_singleclass_img1024/val_batch2_labels.jpg  
  inflating: runs_curetsd_single/runs_curetsd_single/yolov8n_singleclass_img1024/val_batch0_pred.jpg  
  inflating: runs_curetsd_single/runs_curetsd_single/yolov8n_singleclass_img1024/train_batch22050.jpg  
  inflating: runs_curetsd_single/runs_curetsd_single/yolov8n_singleclass_img1024/train_batch1.jpg  
  inflating: runs_curetsd_single/runs_curetsd_single/yolov8n_singleclass_img1024/results.png  
  inflating: runs_curetsd_single/runs_curetsd_single/yolov8n_singleclass_img1024/val_batch0_labels.jpg  
  inflating: runs_curetsd_single/runs_curetsd_s

In [14]:
!unzip  /content/drive/MyDrive/Restormer-Detection-Classfication/Classification/runs_cnn_curetsd.zip

Archive:  /content/drive/MyDrive/Restormer-Detection-Classfication/Classification/runs_cnn_curetsd.zip
   creating: content/runs_cnn_curetsd/
  inflating: content/runs_cnn_curetsd/cnn_last.pt  
  inflating: content/runs_cnn_curetsd/cnn_best.pt  


## Helper Class and Functions

In [4]:

# Basic image helpers
def pad_to_multiple_of_8(x):
    # x: (1,C,H,W)
    _, _, h, w = x.shape
    pad_h = (8 - h % 8) % 8
    pad_w = (8 - w % 8) % 8
    x_pad = F.pad(x, (0, pad_w, 0, pad_h), mode="reflect")
    return x_pad, (h, w)

def unpad(x, orig_hw):
    h, w = orig_hw
    return x[..., :h, :w]

def pil_to_torch01(im_pil: Image.Image):
    arr = np.array(im_pil.convert("RGB"))
    x = torch.from_numpy(arr).permute(2,0,1).float() / 255.0
    return x.unsqueeze(0)  # (1,3,H,W)

def torch01_to_pil(x: torch.Tensor):
    x = x.detach().clamp(0,1)[0].permute(1,2,0).cpu().numpy()
    x = (x * 255.0).round().astype(np.uint8)
    return Image.fromarray(x)

def pil_to_gray01(img: Image.Image, target_size=None) -> np.ndarray:
    if target_size is not None and img.size != target_size:
        img = img.resize(target_size, Image.BILINEAR)
    arr = np.asarray(img.convert("L"), dtype=np.float32) / 255.0
    return arr

def psnr_gray01(a: np.ndarray, b: np.ndarray) -> float:
    mse = float(np.mean((a - b) ** 2))
    if mse == 0:
        return float("inf")
    return 20.0 * math.log10(1.0 / math.sqrt(mse))

def compute_similarity(clean_gt_pil: Image.Image, other_pil: Image.Image):
    """
    PSNR/SSIM computed on grayscale [0,1].
    Resizes other to clean_gt size (standard).
    """
    if clean_gt_pil is None or other_pil is None:
        return {"psnr": None, "ssim": None}
    gt = pil_to_gray01(clean_gt_pil)
    oth = pil_to_gray01(other_pil, target_size=clean_gt_pil.size)
    out = {"psnr": float(psnr_gray01(gt, oth))}
    out["ssim"] = float(ssim_fn(gt, oth, data_range=1.0))
    return out


In [5]:

# GT parsing + IoU + matching
def read_gt_txt(path, W, H, fmt="auto"):
    """
    Reads GT labels and returns list of {"cls": int, "box": [x1,y1,x2,y2]} in pixel coords.

    Supported formats (per line):
      1) Pixel corners (your new GT):
         cls x1 y1 x2 y2 x3 y3 x4 y4

      2) YOLO normalized (legacy):
         cls cx cy w h   (all normalized 0..1)

    fmt:
      - "auto": detect by number of fields (9 -> corners, 5 -> yolo)
      - "corners": force corners format
      - "yolo": force yolo format
    """
    gts = []
    if path is None or (not os.path.exists(path)):
        return gts

    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split()

            # --- auto detect format ---
            if fmt == "auto":
                if len(parts) >= 9:
                    cur_fmt = "corners"
                elif len(parts) >= 5:
                    cur_fmt = "yolo"
                else:
                    continue
            else:
                cur_fmt = fmt

            if cur_fmt == "corners":
                if len(parts) < 9:
                    continue
                cls = int(float(parts[0]))-1
                coords = list(map(float, parts[1:9]))  # 8 numbers

                xs = coords[0::2]
                ys = coords[1::2]
                x1, x2 = min(xs), max(xs)
                y1, y2 = min(ys), max(ys)

            elif cur_fmt == "yolo":
                if len(parts) < 5:
                    continue
                cls = int(float(parts[0]))
                cx, cy, bw, bh = map(float, parts[1:5])

                x1 = (cx - bw / 2) * W
                y1 = (cy - bh / 2) * H
                x2 = (cx + bw / 2) * W
                y2 = (cy + bh / 2) * H
            else:
                raise ValueError(f"Unknown fmt={fmt}")

            # clamp to image bounds
            x1 = max(0, min(W - 1, x1)); x2 = max(0, min(W - 1, x2))
            y1 = max(0, min(H - 1, y1)); y2 = max(0, min(H - 1, y2))

            # drop invalid / degenerate boxes
            if x2 <= x1 or y2 <= y1:
                continue

            gts.append({"cls": cls, "box": [float(x1), float(y1), float(x2), float(y2)]})

    return gts

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    iw = max(0.0, inter_x2 - inter_x1)
    ih = max(0.0, inter_y2 - inter_y1)
    inter = iw * ih
    area_a = max(0.0, ax2-ax1) * max(0.0, ay2-ay1)
    area_b = max(0.0, bx2-bx1) * max(0.0, by2-by1)
    union = area_a + area_b - inter
    return 0.0 if union <= 0 else inter / union

def greedy_match(gts, preds, iou_th=0.5):
    """
    gts: list [{"cls","box"}]
    preds: list [{"box", ...}]
    """
    pairs = []
    for gi, g in enumerate(gts):
        for pi, p in enumerate(preds):
            iou = iou_xyxy(g["box"], p["box"])
            if iou >= iou_th:
                pairs.append((iou, gi, pi))
    pairs.sort(reverse=True, key=lambda x: x[0])

    gt_used, pr_used = set(), set()
    gt_to_pred, pred_to_gt, ious = {}, {}, {}

    for iou, gi, pi in pairs:
        if gi in gt_used or pi in pr_used:
            continue
        gt_used.add(gi)
        pr_used.add(pi)
        gt_to_pred[gi] = pi
        pred_to_gt[pi] = gi
        ious[(gi,pi)] = float(iou)

    return gt_to_pred, pred_to_gt, ious

# Cropping + CNN preprocess
def crop_xyxy(np_img_rgb, box, pad=0):
    h, w = np_img_rgb.shape[:2]
    x1,y1,x2,y2 = box
    x1 = max(0, int(x1) - pad); y1 = max(0, int(y1) - pad)
    x2 = min(w, int(x2) + pad); y2 = min(h, int(y2) + pad)
    if x2 <= x1 or y2 <= y1:
        return None
    return np_img_rgb[y1:y2, x1:x2]

def preprocess_for_cnn(crop_rgb_np, size):
    crop = cv2.resize(crop_rgb_np, (size, size), interpolation=cv2.INTER_AREA)
    x = torch.from_numpy(crop).permute(2,0,1).float() / 255.0
    return x.unsqueeze(0)  # (1,3,size,size)


# Drawing (GT always + FP/FN/MISCLS)
def _put_label(im_bgr, text, x, y, color_bgr, font_scale=0.55, thickness=2):
    font = cv2.FONT_HERSHEY_SIMPLEX
    (tw, th), baseline = cv2.getTextSize(text, font, font_scale, thickness)
    x = int(x); y = int(y)
    H, W = im_bgr.shape[:2]

    x1 = max(0, x - 3)
    y1 = max(0, y - th - 4)
    x2 = min(W-1, x + tw + 6)
    y2 = min(H-1, y + baseline + 4)

    cv2.rectangle(im_bgr, (x1,y1), (x2,y2), (0,0,0), -1)
    cv2.putText(im_bgr, text, (x, y), font, font_scale, color_bgr, thickness, cv2.LINE_AA)

def _draw_rect(im_bgr, box, color_bgr, thickness=2):
    x1,y1,x2,y2 = map(int, box)
    cv2.rectangle(im_bgr, (x1,y1), (x2,y2), color_bgr, thickness)

def _draw_dashed_rect(im_bgr, box, color_bgr, thickness=2, dash_len=10, gap=6):
    x1,y1,x2,y2 = map(int, box)

    def dashed_line(p1, p2):
        x1l, y1l = p1
        x2l, y2l = p2
        length = int(np.hypot(x2l-x1l, y2l-y1l))
        if length <= 0:
            return
        vx = (x2l - x1l) / length
        vy = (y2l - y1l) / length
        d = 0
        while d < length:
            d2 = min(d + dash_len, length)
            sx = int(x1l + vx * d);   sy = int(y1l + vy * d)
            ex = int(x1l + vx * d2);  ey = int(y1l + vy * d2)
            cv2.line(im_bgr, (sx,sy), (ex,ey), color_bgr, thickness)
            d += dash_len + gap

    dashed_line((x1,y1), (x2,y1))
    dashed_line((x2,y1), (x2,y2))
    dashed_line((x2,y2), (x1,y2))
    dashed_line((x1,y2), (x1,y1))

def draw_eval(np_img_rgb, gts, preds, gt_to_pred, pred_to_gt, ious, class_names=None):
    """
    Colors:
      GT: Blue
      FN: Blue + 'FN MISS' label (cyan text)
      FP: Red dashed
      TP: Green
      MISCLS: Orange (label shows GT vs Pred)
    """
    im = cv2.cvtColor(np_img_rgb, cv2.COLOR_RGB2BGR)
    H, W = im.shape[:2]

    #gt_to_pred = [x - 1 for x in gt_to_pred]

    def cname(cls_id):
        cls_id = int(cls_id)
        if class_names and 0 <= cls_id < len(class_names):
            return f"{cls_id}:{class_names[cls_id]}"
        return str(cls_id)

    # GT always
    for gi, g in enumerate(gts):
        x1,y1,x2,y2 = map(int, g["box"])
        gt_name = cname(g["cls"])

        if gi in gt_to_pred:
            pi = gt_to_pred[gi]
            iou = ious.get((gi,pi), 0.0)
            _draw_rect(im, (x1,y1,x2,y2), (255,0,0), thickness=2)
            _put_label(im, f"GT {gt_name} | IoU {iou:.2f}", x1, max(18, y1-6), (255,0,0))
        else:
            _draw_rect(im, (x1,y1,x2,y2), (255,0,0), thickness=3)
            txt = f"FN MISS | GT={gt_name}"
            y_lbl = max(18, y1-6) if (y2 + 24 >= H) else min(H-6, y2 + 18)
            _put_label(im, txt, x1, y_lbl, (255,255,0), font_scale=0.60, thickness=2)

    # Predictions
    for pi, p in enumerate(preds):
        b = p["box"]
        x1,y1,x2,y2 = map(int, b)
        det_name = str(p.get("det_name","det"))
        cnn_name = str(p.get("cnn_name","cls"))
        conf = float(p.get("conf",0.0))

        if pi not in pred_to_gt:
            _draw_dashed_rect(im, (x1,y1,x2,y2), (0,0,255), thickness=2)
            _put_label(im, f"FP | {det_name}|{cnn_name} | det={conf:.2f}", x1, max(18, y1-6), (0,0,255))
        else:
            gi = pred_to_gt[pi]
            gt_cls = gts[gi]["cls"]
            ok = (int(p["cnn_cls"]) == int(gt_cls))
            if ok:
                color = (0,255,0)
                txt = f"TP | {det_name}|{cnn_name} | det={conf:.2f}"
            else:
                color = (0,165,255)
                txt = f"MISCLS | GT={cname(gt_cls)} | Pred={cnn_name} | det={conf:.2f}"
            _draw_rect(im, (x1,y1,x2,y2), color, thickness=3)
            _put_label(im, txt, x1, max(18, y1-6), color)

    return cv2.cvtColor(im, cv2.COLOR_BGR2RGB)

In [6]:

# 4) Table builder
def build_table(gts, preds, gt_to_pred, pred_to_gt, ious, class_names=None):
    def cname(cls_id):
        cls_id = int(cls_id)
        if class_names and 0 <= cls_id < len(class_names):
            return f"{cls_id}:{class_names[cls_id]}"
        return str(cls_id)

    rows = []

    # GT-centric rows (FN always visible)
    for gi, g in enumerate(gts):
        gt_cls = g["cls"]
        gt_box = g["box"]
        if gi in gt_to_pred:
            pi = gt_to_pred[gi]
            p = preds[pi]
            iou = float(ious.get((gi,pi), 0.0))
            status = "TP" if int(p["cnn_cls"]) == int(gt_cls) else "MISCLS"
            rows.append({
                "status": status,
                "gt_class": cname(gt_cls),
                "pred_class": str(p.get("cnn_name", p.get("cnn_cls"))),
                "iou": iou,
                "det_conf": float(p.get("conf", np.nan)),
                "cnn_prob": float(p.get("cnn_prob", np.nan)),
                "x1": float(gt_box[0]), "y1": float(gt_box[1]), "x2": float(gt_box[2]), "y2": float(gt_box[3]),
            })
        else:
            rows.append({
                "status": "FN",
                "gt_class": cname(gt_cls),
                "pred_class": "",
                "iou": 0.0,
                "det_conf": np.nan,
                "cnn_prob": np.nan,
                "x1": float(gt_box[0]), "y1": float(gt_box[1]), "x2": float(gt_box[2]), "y2": float(gt_box[3]),
            })

    # FP rows
    for pi, p in enumerate(preds):
        if pi not in pred_to_gt:
            b = p["box"]
            rows.append({
                "status": "FP",
                "gt_class": "",
                "pred_class": str(p.get("cnn_name", p.get("cnn_cls"))),
                "iou": 0.0,
                "det_conf": float(p.get("conf", np.nan)),
                "cnn_prob": float(p.get("cnn_prob", np.nan)),
                "x1": float(b[0]), "y1": float(b[1]), "x2": float(b[2]), "y2": float(b[3]),
            })

    df = pd.DataFrame(rows, columns=["status","gt_class","pred_class","iou","det_conf","cnn_prob","x1","y1","x2","y2"])
    # Sort: FN first, then MISCLS, then TP, then FP
    order = {"FN": 0, "MISCLS": 1, "TP": 2, "FP": 3}
    df["_ord"] = df["status"].map(lambda s: order.get(str(s), 99))
    df = df.sort_values(["_ord", "det_conf"], ascending=[True, False], na_position="last").drop(columns=["_ord"])
    return df

## Models

### Restormer

In [7]:
import numbers
from einops import rearrange


In [8]:
# Reshape helpers for LayerNorm
def to_3d(x):
    return rearrange(x, 'b c h w -> b (h w) c')

def to_4d(x, h, w):
    return rearrange(x, 'b (h w) c -> b c h w', h=h, w=w)

# LayerNorm without mean subtraction
class BiasFree_LayerNorm(nn.Module):
    def __init__(self, normalized_shape):
        super().__init__()
        if isinstance(normalized_shape, numbers.Integral):
            normalized_shape = (normalized_shape,)
        normalized_shape = torch.Size(normalized_shape)
        assert len(normalized_shape) == 1
        self.weight = nn.Parameter(torch.ones(normalized_shape))  # scale only

    def forward(self, x):
        sigma = x.var(-1, keepdim=True, unbiased=False)
        return x / torch.sqrt(sigma + 1e-5) * self.weight

# Standard LayerNorm with mean and bias
class WithBias_LayerNorm(nn.Module):
    def __init__(self, normalized_shape):
        super().__init__()
        if isinstance(normalized_shape, numbers.Integral):
            normalized_shape = (normalized_shape,)
        normalized_shape = torch.Size(normalized_shape)
        assert len(normalized_shape) == 1
        self.weight = nn.Parameter(torch.ones(normalized_shape))    # scale
        self.bias = nn.Parameter(torch.zeros(normalized_shape))     # shift

    def forward(self, x):
        mu = x.mean(-1, keepdim=True)   # channel mean
        sigma = x.var(-1, keepdim=True, unbiased=False)
        return (x - mu) / torch.sqrt(sigma + 1e-5) * self.weight + self.bias

# LayerNorm wrapper for 2D feature maps
class LayerNorm(nn.Module):
    def __init__(self, dim, LayerNorm_type='WithBias'):
        super().__init__()
        self.body = BiasFree_LayerNorm(dim) if LayerNorm_type == 'BiasFree' else WithBias_LayerNorm(dim)

    def forward(self, x):
        h, w = x.shape[-2:]      # save spatial size
        return to_4d(self.body(to_3d(x)), h, w)     # norm over channels


class FeedForward(nn.Module):
    def __init__(self, dim, ffn_expansion_factor=2.66, bias=False):
        super().__init__()
        hidden_features = int(dim * ffn_expansion_factor)   # channel expansion
        self.project_in  = nn.Conv2d(dim, hidden_features * 2, 1, bias=bias)    # expand + gate
        self.dwconv      = nn.Conv2d(hidden_features * 2, hidden_features * 2, 3, 1, 1,
                                     groups=hidden_features * 2, bias=bias) # depthwise conv
        self.project_out = nn.Conv2d(hidden_features, dim, 1, bias=bias)    # project back

    def forward(self, x):
        x = self.project_in(x)                      # channel expansion
        x1, x2 = self.dwconv(x).chunk(2, dim=1)     # gated split
        x = F.gelu(x1) * x2                         # gated activation
        return self.project_out(x)                  # channel reduction



class Attention(nn.Module):
    def __init__(self, dim, num_heads=1, bias=False):
        super().__init__()
        self.num_heads = num_heads
        self.temperature = nn.Parameter(torch.ones(num_heads, 1, 1)) # per-head scale

        self.qkv        = nn.Conv2d(dim, dim * 3, 1, bias=bias)     # QKV projection
        self.qkv_dwconv = nn.Conv2d(dim * 3, dim * 3, 3, 1, 1, groups=dim * 3, bias=bias)  # spatial mixing
        self.project_out = nn.Conv2d(dim, dim, 1, bias=bias)    # output proj

    def forward(self, x):
        b, c, h, w = x.shape

        qkv = self.qkv_dwconv(self.qkv(x)) # generate Q,K,V
        q, k, v = qkv.chunk(3, dim=1)   # split

        q = rearrange(q, 'b (head c) h w -> b head c (h w)', head=self.num_heads)
        k = rearrange(k, 'b (head c) h w -> b head c (h w)', head=self.num_heads)
        v = rearrange(v, 'b (head c) h w -> b head c (h w)', head=self.num_heads)

        q = F.normalize(q, dim=-1)  # L2 normalize
        k = F.normalize(k, dim=-1)  # L2 normalize

        attn = (q @ k.transpose(-2, -1)) * self.temperature     # attention
        attn = attn.softmax(dim=-1)

        out = (attn @ v)            # weighted sum
        out = rearrange(out, 'b head c (h w) -> b (head c) h w', head=self.num_heads, h=h, w=w)

        return self.project_out(out)    # final projection


class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, ffn_expansion_factor=2.66, bias=False, LayerNorm_type='WithBias'):
        super().__init__()
        self.norm1 = LayerNorm(dim, LayerNorm_type)   # pre-attention norm
        self.attn  = Attention(dim, num_heads, bias)  # self-attention
        self.norm2 = LayerNorm(dim, LayerNorm_type)   # pre-FFN norm
        self.ffn   = FeedForward(dim, ffn_expansion_factor, bias)  # feed-forward

    def forward(self, x):
        x = x + self.attn(self.norm1(x))  # attention + residual
        x = x + self.ffn(self.norm2(x))   # FFN + residual
        return x

class OverlapPatchEmbed(nn.Module):
    def __init__(self, in_c=3, embed_dim=48, bias=False):
        super().__init__()
        self.proj = nn.Conv2d(in_c, embed_dim, 3, 1, 1, bias=bias)  # overlapping patch embed

    def forward(self, x):
        return self.proj(x) # project image to feature space


class Downsample(nn.Module):
    def __init__(self, n_feat):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(n_feat, n_feat // 2, 3, 1, 1, bias=False), # channel reduce
            nn.PixelUnshuffle(2)  # spatial downsample
            )

    def forward(self, x):
        return self.body(x) # downsample feature map


class Upsample(nn.Module):
    def __init__(self, n_feat):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(n_feat, n_feat * 2, 3, 1, 1, bias=False), # channel expand
            nn.PixelShuffle(2) # spatial upsample
        )

    def forward(self, x):
        return self.body(x) # upsample feature map


class Restormer(nn.Module):
    def __init__(
        self,
        inp_channels=3,
        out_channels=3,
        dim=48,
        num_blocks=[4,6,6,8],
        num_refinement_blocks=4,
        heads=[1,2,4,8],
        ffn_expansion_factor=2.66,
        bias=False,
        LayerNorm_type='WithBias',
        dual_pixel_task=False
    ):
        super().__init__()

        # patch embedding (no stride): (B, inp_channels, H, W) -> (B, dim, H, W)
        self.patch_embed = OverlapPatchEmbed(inp_channels, dim)

        # encoder level 1 (full resolution)
        self.encoder_level1 = nn.Sequential(*[
            TransformerBlock(dim, heads[0], ffn_expansion_factor, bias, LayerNorm_type)
            for _ in range(num_blocks[0])
        ])

        # encoder level 2 (H/2, W/2)
        self.down1_2 = Downsample(dim)
        self.encoder_level2 = nn.Sequential(*[
            TransformerBlock(int(dim*2**1), heads[1], ffn_expansion_factor, bias, LayerNorm_type)
            for _ in range(num_blocks[1])
        ])

        # encoder level 3 (H/4, W/4)
        self.down2_3 = Downsample(int(dim*2**1))
        self.encoder_level3 = nn.Sequential(*[
            TransformerBlock(int(dim*2**2), heads[2], ffn_expansion_factor, bias, LayerNorm_type)
            for _ in range(num_blocks[2])
        ])

        # encoder level 4 / latent (H/8, W/8)
        self.down3_4 = Downsample(int(dim*2**2))
        self.latent = nn.Sequential(*[
            TransformerBlock(int(dim*2**3), heads[3], ffn_expansion_factor, bias, LayerNorm_type)
            for _ in range(num_blocks[3])
        ])

        # decoder level 3 (upsample + skip from enc3)
        self.up4_3 = Upsample(int(dim*2**3))
        self.reduce_chan_level3 = nn.Conv2d(int(dim*2**3), int(dim*2**2), 1, bias=bias)
        self.decoder_level3 = nn.Sequential(*[
            TransformerBlock(int(dim*2**2), heads[2], ffn_expansion_factor, bias, LayerNorm_type)
            for _ in range(num_blocks[2])
        ])

        # decoder level 2 (upsample + skip from enc2)
        self.up3_2 = Upsample(int(dim*2**2))
        self.reduce_chan_level2 = nn.Conv2d(int(dim*2**2), int(dim*2**1), 1, bias=bias)
        self.decoder_level2 = nn.Sequential(*[
            TransformerBlock(int(dim*2**1), heads[1], ffn_expansion_factor, bias, LayerNorm_type)
            for _ in range(num_blocks[1])
        ])

        # decoder level 1 (upsample + skip from enc1)
        self.up2_1 = Upsample(int(dim*2**1))
        self.decoder_level1 = nn.Sequential(*[
            TransformerBlock(int(dim*2**1), heads[0], ffn_expansion_factor, bias, LayerNorm_type)
            for _ in range(num_blocks[0])
        ])

        # refinement blocks at full resolution
        self.refinement = nn.Sequential(*[
            TransformerBlock(int(dim*2**1), heads[0], ffn_expansion_factor, bias, LayerNorm_type)
            for _ in range(num_refinement_blocks)
        ])

        # optional dual-pixel skip
        self.dual_pixel_task = dual_pixel_task
        if self.dual_pixel_task:
            self.skip_conv = nn.Conv2d(dim, int(dim*2**1), 1, bias=bias)

        # output projection
        self.output = nn.Conv2d(int(dim*2**1), out_channels, 3, 1, 1, bias=bias)

    def forward(self, inp_img):
        # ---- Encoder ----
        inp_enc_level1 = self.patch_embed(inp_img)
        out_enc_level1 = self.encoder_level1(inp_enc_level1)

        inp_enc_level2 = self.down1_2(out_enc_level1)
        out_enc_level2 = self.encoder_level2(inp_enc_level2)

        inp_enc_level3 = self.down2_3(out_enc_level2)
        out_enc_level3 = self.encoder_level3(inp_enc_level3)

        inp_enc_level4 = self.down3_4(out_enc_level3)
        latent = self.latent(inp_enc_level4)

        # ---- Decoder (with skip connections) ----
        inp_dec_level3 = self.up4_3(latent)
        inp_dec_level3 = torch.cat([inp_dec_level3, out_enc_level3], 1) # concat skip
        inp_dec_level3 = self.reduce_chan_level3(inp_dec_level3)        # reduce after concat
        out_dec_level3 = self.decoder_level3(inp_dec_level3)

        inp_dec_level2 = self.up3_2(out_dec_level3)
        inp_dec_level2 = torch.cat([inp_dec_level2, out_enc_level2], 1) # concat skip
        inp_dec_level2 = self.reduce_chan_level2(inp_dec_level2)
        out_dec_level2 = self.decoder_level2(inp_dec_level2)

        inp_dec_level1 = self.up2_1(out_dec_level2)
        inp_dec_level1 = torch.cat([inp_dec_level1, out_enc_level1], 1) # concat skip
        out_dec_level1 = self.decoder_level1(inp_dec_level1)

        # final refinement at full resolution
        out_dec_level1 = self.refinement(out_dec_level1)

        # ---- Output head ----
        if self.dual_pixel_task:
            out_dec_level1 = out_dec_level1 + self.skip_conv(inp_enc_level1)    # extra skip-add
            out_dec_level1 = self.output(out_dec_level1)                        # direct output
        else:
            out_dec_level1 = self.output(out_dec_level1) + inp_img  # global residual

        return out_dec_level1





In [15]:
def load_restormer_checkpoint(model, ckpt_path, strict=False):
    # load checkpoint to CPU (safe for different devices / PyTorch 2.x)
    ckpt = torch.load(
        ckpt_path,
        map_location="cpu",
        weights_only=False  # IMPORTANT for PyTorch 2.x
    )

    if isinstance(ckpt, dict):
        if "params" in ckpt:
            state = ckpt["params"]  # Restormer-style checkpoint
        elif "state_dict" in ckpt:
            state = ckpt["state_dict"]  # common PyTorch format
        else:
            state = ckpt # raw state_dict
    else:
        state = ckpt

    missing, unexpected = model.load_state_dict(state, strict=strict)
    print(f"Loaded {ckpt_path} | missing={len(missing)} unexpected={len(unexpected)}")

def pick_ckpt_for_subset(subset_name: str):
    s = subset_name.lower()
    if "rain" in s:
        return CKPT_RAIN
    if "exposure" in s:
        return CKPT_EXPOSURE
    if "blur" in s:  # handles lens_blur + gaussian_blur
        return CKPT_BLUR
    return None  # clean -> copy as-is

## MiniCNN

In [16]:
class MiniCNN(nn.Module):
    """
    MiniCNN is a compact baseline CNN classifier designed for small image crops
    (e.g., 64×64 traffic sign patches). It prioritizes speed and training stability
    while maintaining enough representational capacity to separate 14 classes.
    """
    def __init__(self, num_classes=14):
        super().__init__()

        def block(in_ch, out_ch):
            # Two 3×3 convolutions for local feature extraction,
            # followed by spatial downsampling via max pooling
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),  # stabilizes training across batches
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2)         # halves spatial resolution
            )

        self.features = nn.Sequential(
            block(3, 32),    # 64->32
            block(32, 64),   # 32->16
            nn.Sequential(   # 16->8
                nn.Conv2d(64, 128, 3, padding=1, bias=False),
                nn.BatchNorm2d(128),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2)
            )
        )
        # Global average pooling removes spatial dependence and reduces parameters
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        # Dropout improves generalization on small cropped datasets
        self.dropout = nn.Dropout(p=0.30)
        # Final linear classifier mapping features to class logits
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x).flatten(1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

### Loading Model Checkpoints

In [20]:
PROJECT_ROOT = Path(".").resolve()

# Trained weights
YOLO_WEIGHTS = PROJECT_ROOT / "runs_curetsd_single" / "yolov8n_singleclass_img1024" / "weights" / "best.pt"
CNN_WEIGHTS  = PROJECT_ROOT / "content" /"runs_cnn_curetsd" / "cnn_best.pt"


# Where your Restormer .pth files live
RESTORMERS_ROOT = Path("/content/drive/MyDrive/Restormer-Detection-Classfication/Restormers/Fine-Tuned")


assert YOLO_WEIGHTS.exists(), f"Missing: {YOLO_WEIGHTS}"
assert CNN_WEIGHTS.exists(), f"Missing: {CNN_WEIGHTS}"
assert RESTORMERS_ROOT.exists(), f"Missing: {RESTORMERS_ROOT}"



# ---- Devices ----
def pick_device():
    if torch.cuda.is_available():
        return 0
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

YOLO_DEVICE = pick_device()
CNN_DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if (getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()) else "cpu"))


In [21]:
# ---- YOLO ----
model = YOLO("yolov8s.pt")
yolo = YOLO(str(YOLO_WEIGHTS))
print("Loaded YOLO weights:", YOLO_WEIGHTS)


# ---- Load checkpoint ----
ckpt = torch.load(CNN_WEIGHTS, map_location="cpu")
CNN_IMG_SIZE = int(ckpt.get("img_size", 64))
CNN_NUM_CLASSES = int(ckpt.get("num_classes", 14))

cnn = MiniCNN(num_classes=CNN_NUM_CLASSES)
cnn.load_state_dict(ckpt["model_state"])
cnn = cnn.to(CNN_DEVICE)
cnn.eval()

print("Loaded CNN weights:", CNN_WEIGHTS)
print("CNN input size:", CNN_IMG_SIZE, "| num_classes:", CNN_NUM_CLASSES)


Loaded YOLO weights: /content/runs_curetsd_single/yolov8n_singleclass_img1024/weights/best.pt
Loaded CNN weights: /content/content/runs_cnn_curetsd/cnn_best.pt
CNN input size: 64 | num_classes: 14


In [22]:
CKPT_BLUR     = RESTORMERS_ROOT / "blur_curetsd.pth"
CKPT_RAIN     = RESTORMERS_ROOT / "rain_curetsd.pth"
CKPT_EXPOSURE = RESTORMERS_ROOT / "exposure_curetsd.pth"

assert CKPT_BLUR.exists(), f"Missing CKPT_BLUR: {CKPT_BLUR}"
assert CKPT_RAIN.exists(), f"Missing CKPT_RAIN: {CKPT_RAIN}"
assert CKPT_EXPOSURE.exists(), f"Missing CKPT_EXPOSURE: {CKPT_EXPOSURE}"


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


MODEL_CFG = dict(
    inp_channels=3,
    out_channels=3,
    dim=48,
    num_blocks=[4, 6, 6, 8],
    num_refinement_blocks=4,
    heads=[1, 2, 4, 8],
    ffn_expansion_factor=2.66
)

# Create and load three separate models (recommended; weights differ)
model_blur = Restormer(**MODEL_CFG).to(device).eval()
model_rain = Restormer(**MODEL_CFG).to(device).eval()
model_exp  = Restormer(**MODEL_CFG).to(device).eval()

load_restormer_checkpoint(model_blur, CKPT_BLUR, strict=False)
load_restormer_checkpoint(model_rain, CKPT_RAIN, strict=False)
load_restormer_checkpoint(model_exp,  CKPT_EXPOSURE, strict=False)

model_map = {
    str(CKPT_BLUR): model_blur,
    str(CKPT_RAIN): model_rain,
    str(CKPT_EXPOSURE): model_exp,
}

Device: cuda
Loaded /content/drive/MyDrive/Restormer-Detection-Classfication/Restormers/Fine-Tuned/blur_curetsd.pth | missing=0 unexpected=0
Loaded /content/drive/MyDrive/Restormer-Detection-Classfication/Restormers/Fine-Tuned/rain_curetsd.pth | missing=0 unexpected=0
Loaded /content/drive/MyDrive/Restormer-Detection-Classfication/Restormers/Fine-Tuned/exposure_curetsd.pth | missing=0 unexpected=0


## Pipeline

In [23]:
# Restoration selection wrapper


CLASS_NAMES = [
    "speed_limit", "goods_vehicles", "no_overtaking", "no_stopping",
    "no_parking", "stop", "bicycle", "hump",
    "no_left", "no_right", "priority_to", "no_entry",
    "yield", "parking"
]

try:
    CLASS_NAMES
except NameError:
    CLASS_NAMES = None

restoration_map = {
    "None (skip restoration)": None,
    "Blur": model_blur,
    "Rain": model_rain,
    "Exposure": model_exp,
}

@torch.no_grad()
def run_restoration(img_pil: Image.Image, mode: str):
    if img_pil is None:
        return None
    img_pil = img_pil.convert("RGB")
    model = restoration_map.get(mode, None)
    if model is None:
        return img_pil

    x = pil_to_torch01(img_pil).to(device)
    x_pad, orig_hw = pad_to_multiple_of_8(x)
    y = model(x_pad)
    y = unpad(y, orig_hw)
    return torch01_to_pil(y)


In [24]:

# Full evaluation pipeline for Gradio
@torch.no_grad()
def gradio_pipeline(
    degraded_img_pil,
    clean_gt_img_pil,     # optional
    gt_label_file,        # optional .txt
    restoration_mode,
    det_conf,
    det_iou,
    match_iou_th,
    crop_pad
):
    if degraded_img_pil is None:
        return None, None, pd.DataFrame(), {}

    degraded_img_pil = degraded_img_pil.convert("RGB")

    # (1) Restore
    restored_pil = run_restoration(degraded_img_pil, restoration_mode)

    # (2) PSNR/SSIM vs clean GT (optional)
    metrics = {}
    if clean_gt_img_pil is not None:
        clean_gt_img_pil = clean_gt_img_pil.convert("RGB")
        sim = compute_similarity(clean_gt_img_pil, restored_pil)
        metrics["PSNR(restored vs cleanGT)"] = sim["psnr"]
        metrics["SSIM(restored vs cleanGT)"] = sim["ssim"]

    # (3) Load GT labels (optional)
    W, H = restored_pil.size
    gts = []
    if gt_label_file is not None:
        gt_path = gt_label_file.name if hasattr(gt_label_file, "name") else str(gt_label_file)
        gts = read_gt_txt(gt_path, W, H)

    # (4) Detect on restored
    res = yolo.predict(restored_pil, conf=float(det_conf), iou=float(det_iou), verbose=False)[0]

    preds = []
    restored_np = np.array(restored_pil)

    if res.boxes is not None and len(res.boxes) > 0:
        boxes = res.boxes.xyxy.detach().cpu().numpy()
        confs = res.boxes.conf.detach().cpu().numpy()
        detcls = res.boxes.cls.detach().cpu().numpy().astype(int)

        for i, (b, c, dc) in enumerate(zip(boxes, confs, detcls)):
            crop = crop_xyxy(restored_np, b, pad=int(crop_pad))
            if crop is None or crop.size == 0:
                continue

            x = preprocess_for_cnn(crop, CNN_IMG_SIZE).to(CNN_DEVICE)
            logits = cnn(x)
            probs = torch.softmax(logits, dim=1)[0]
            cnn_cls = int(probs.argmax().item())
            cnn_prob = float(probs[cnn_cls].item())

            det_name = yolo.names.get(int(dc), str(int(dc)))
            if CLASS_NAMES and 0 <= cnn_cls < len(CLASS_NAMES):
                cnn_name = f"{cnn_cls}:{CLASS_NAMES[cnn_cls]}"
            else:
                cnn_name = str(cnn_cls)

            preds.append({
                "pred_idx": int(i),
                "box": [float(b[0]), float(b[1]), float(b[2]), float(b[3])],
                "conf": float(c),
                "det_cls": int(dc),
                "det_name": str(det_name),
                "cnn_cls": int(cnn_cls),
                "cnn_name": str(cnn_name),
                "cnn_prob": float(cnn_prob)
            })

    # (5) Match with GT (optional)
    gt_to_pred, pred_to_gt, ious = greedy_match(gts, preds, iou_th=float(match_iou_th))

    # (6) Metrics (detection + cls on matched)
    TP = len(gt_to_pred)
    FP = max(0, len(preds) - len(pred_to_gt))
    FN = max(0, len(gts) - len(gt_to_pred))
    MISCLS = 0
    correct_cls = 0
    for gi, pi in gt_to_pred.items():
        if int(preds[pi]["cnn_cls"]) == int(gts[gi]["cls"]):
            correct_cls += 1
        else:
            MISCLS += 1

    metrics.update({
        "GT count": int(len(gts)),
        "Pred count": int(len(preds)),
        "TP(matched GT)": int(TP),
        "FP(unmatched pred)": int(FP),
        "FN(missed GT)": int(FN),
        "Misclassified (matched)": int(MISCLS),
        "Det Precision": float(TP / (TP + FP + 1e-9)),
        "Det Recall": float(TP / (TP + FN + 1e-9)),
        "Cls Acc (matched)": float(correct_cls / (TP + 1e-9)) if TP > 0 else 0.0
    })

    # (7) Draw overlay (GT always)
    annotated_rgb = draw_eval(restored_np, gts, preds, gt_to_pred, pred_to_gt, ious, class_names=CLASS_NAMES)
    annotated_pil = Image.fromarray(annotated_rgb)

    # (8) Table
    df_table = build_table(gts, preds, gt_to_pred, pred_to_gt, ious, class_names=CLASS_NAMES)

    return restored_pil, annotated_pil, df_table, metrics

## GUI

In [ ]:
# ============================================================
# GRADIO UI
# ============================================================
with gr.Blocks() as demo:
    gr.Markdown("# Restoration → Detection → Classification (GT-aware + PSNR/SSIM)")

    with gr.Row():
        inp_degraded = gr.Image(type="pil", label="Degraded Input (restore + detect)")
        inp_clean_gt = gr.Image(type="pil", label="Clean GT Image (optional, for PSNR/SSIM)")

    gt_file = gr.File(label="GT Labels (.txt) YOLO format: class cx cy w h (optional)", file_types=[".txt"])

    with gr.Row():
        restoration_mode = gr.Dropdown(
            choices=list(restoration_map.keys()),
            value="Blur",
            label="Restoration Model"
        )

    with gr.Row():
        det_conf = gr.Slider(0.001, 0.95, value=0.25, step=0.001, label="YOLO Conf")
        det_iou  = gr.Slider(0.10, 0.95, value=0.50, step=0.01, label="YOLO IoU")
        match_iou_th = gr.Slider(0.10, 0.95, value=0.50, step=0.01, label="GT Matching IoU Threshold")
        crop_pad = gr.Slider(0, 30, value=4, step=1, label="Crop Padding (px)")

    run_btn = gr.Button("Run")

    with gr.Row():
        out_restored = gr.Image(type="pil", label="Restored Output")
        out_annot    = gr.Image(type="pil", label="Annotated (GT + TP/FP/FN/MISCLS)")

    metrics_out = gr.JSON(label="Metrics")
    table = gr.Dataframe(label="Results Table (FN/FP/MISCLS included)")

    run_btn.click(
        fn=gradio_pipeline,
        inputs=[inp_degraded, inp_clean_gt, gt_file, restoration_mode, det_conf, det_iou, match_iou_th, crop_pad],
        outputs=[out_restored, out_annot, table, metrics_out]
    )

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4e4775f84187e012ba.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
